# NO-SEED MODEL — stats only, seeds excluded from features

In [3]:
import ast
import json
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier, XGBRegressor

warnings.simplefilter(action="ignore", category=UserWarning)

In [4]:
final = (
    pd.read_csv('data_2026/final_features_W.csv')
    .rename(columns={
        'WTEAMID': 'W_TEAMID',
        'LTEAMID': 'L_TEAMID',
        'WSCORE': 'W_SCORE',
        'LSCORE': 'L_SCORE',
    })
)
# W_TEAMID is always the winner in historical tourney data
final['WIN_INDICATOR'] = 1

C:\Users\joebu\AppData\Local\Temp\ipykernel_31904\1455663223.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final['WIN_INDICATOR'] = 1


In [5]:
# Columns with null values and their respective counts
{col: int(final[col].isna().sum()) for col in final.columns if final[col].isna().any()}

{}

In [6]:
drop_cols = ['W_CTWINS', 'W_AVERAGECTSCORE', 'L_CTWINS', 'L_AVERAGECTSCORE']
final = final.drop(columns=[c for c in drop_cols if c in final.columns])
final = final.drop(columns=[c for c in ['W_WLOCN','W_WLOCH','W_WLOCA','L_WLOCN','L_WLOCH','L_WLOCA'] if c in final.columns])  # variants

In [7]:
parameters = {
    "n_estimators": [100, 200, 300, 400, 500],
    "learning_rate": [0.1, 0.2, 0.3, 0.4, 0.5],
    "max_depth": list(range(3, 6, 1)),
    "min_child_weight": list(range(1, 6, 1)),
}

In [8]:
# W_SEED and L_SEED excluded so the model must rely entirely on season stats
NON_FEATURE_COLS = {'SEASON', 'WIN_INDICATOR', 'L_TEAMID', 'W_TEAMID', 'W_SCORE', 'L_SCORE',
                    'ROUND', 'L_REGION', 'W_REGION', 'W_SEED', 'L_SEED'}
feature_cols = [c for c in final.columns if c not in NON_FEATURE_COLS]

train = final[(final.SEASON >= 2010) & (final.SEASON <= 2023)].copy()
test = final[final.SEASON > 2023].copy()

BRACKET_SEASON = 2026  # Season to predict the bracket for

### Swap the W and L teams

In [9]:
# Double training data with both W/L orientations — perfectly balanced, reproducible, 2x data
df = train.copy()
w_cols = sorted([c for c in df.columns if c.startswith('W_')])
l_cols = ['L_' + c[2:] for c in w_cols]

df_swapped = df.copy()
for w, l in zip(w_cols, l_cols):
    df_swapped[w] = df[l]
    df_swapped[l] = df[w]

train = pd.concat([df, df_swapped], ignore_index=True)
train['WIN_INDICATOR'] = (train['W_SCORE'] > train['L_SCORE']).astype(int)

In [10]:
X_train = train[feature_cols]
y_train = train['WIN_INDICATOR']

all_rounds = GridSearchCV(
    estimator=XGBClassifier(eval_metric='logloss'),
    param_grid=parameters,
    n_jobs=-1,
    scoring="accuracy",
    cv=5,
)
all_rounds.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.1, 0.2, ...], 'max_depth': [3, 4, ...], 'min_child_weight': [1, 2, ...], 'n_estimators': [100, 200, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold an

### Train spread and total score models

In [11]:
# Spread = W_SCORE - L_SCORE (positive when W team wins, negative when L team wins)
# Total = W_SCORE + L_SCORE
y_train_spread = train['W_SCORE'] - train['L_SCORE']
y_train_total  = train['W_SCORE'] + train['L_SCORE']

spread_model_cv = GridSearchCV(
    estimator=XGBRegressor(eval_metric='rmse'),
    param_grid=parameters,
    n_jobs=-1,
    scoring='neg_mean_squared_error',
    cv=5,
)
spread_model_cv.fit(X_train, y_train_spread)
print(f'Spread model best params: {spread_model_cv.best_params_}')

total_model_cv = GridSearchCV(
    estimator=XGBRegressor(eval_metric='rmse'),
    param_grid=parameters,
    n_jobs=-1,
    scoring='neg_mean_squared_error',
    cv=5,
)
total_model_cv.fit(X_train, y_train_total)
print(f'Total model best params: {total_model_cv.best_params_}')

Spread model best params: {'learning_rate': 0.1, 'max_depth': 3, 'min_child_weight': 3, 'n_estimators': 100}
Total model best params: {'learning_rate': 0.1, 'max_depth': 3, 'min_child_weight': 2, 'n_estimators': 100}


In [12]:
def swap_wl(df):
    """Return a copy of df with all W_ and L_ columns swapped, preserving dtypes."""
    out = df.copy()
    w_cols = sorted([c for c in df.columns if c.startswith('W_')])
    l_cols = ['L_' + c[2:] for c in w_cols]
    for w, l in zip(w_cols, l_cols):
        out[w] = df[l]
        out[l] = df[w]
    return out

# Unbiased evaluation: each game tested in both orientations, WIN_INDICATOR derived from score
test_r1 = test[test.ROUND == 1].copy()
test_eval = pd.concat([test_r1, swap_wl(test_r1)], ignore_index=True)
test_eval['WIN_INDICATOR'] = (test_eval['W_SCORE'] > test_eval['L_SCORE']).astype(int)

test_eval['PRED_WIN_INDICATOR'] = all_rounds.predict(test_eval[feature_cols])

for season in sorted(test_eval.SEASON.unique()):
    acc = accuracy_score(
        test_eval[test_eval.SEASON == season]['WIN_INDICATOR'],
        test_eval[test_eval.SEASON == season]['PRED_WIN_INDICATOR'],
    )
    print(f'Accuracy {season}: {acc:.4f}')
    globals()[f'accuracy_{season}'] = acc

accuracy = accuracy_score(test_eval['WIN_INDICATOR'], test_eval['PRED_WIN_INDICATOR'])
print(f'Accuracy total: {accuracy:.4f}')

Accuracy 2024: 0.7903
Accuracy 2025: 0.7969
Accuracy total: 0.7937


C:\Users\joebu\AppData\Local\Temp\ipykernel_31904\4249703926.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_eval['PRED_WIN_INDICATOR'] = all_rounds.predict(test_eval[feature_cols])


#### Save the trained model

In [13]:
optimal_spread_model = spread_model_cv.best_estimator_
optimal_total_model  = total_model_cv.best_estimator_

joblib.dump(optimal_spread_model, 'model/spread_model_no_seed_W.joblib')
joblib.dump(optimal_total_model,  'model/total_model_no_seed_W.joblib')
print('Spread and total (no-seed) models saved.')

# Evaluate on test set
test_eval['PRED_SPREAD'] = optimal_spread_model.predict(test_eval[feature_cols])
test_eval['PRED_TOTAL']  = optimal_total_model.predict(test_eval[feature_cols])
test_eval['ACTUAL_SPREAD'] = test_eval['W_SCORE'] - test_eval['L_SCORE']
test_eval['ACTUAL_TOTAL']  = test_eval['W_SCORE'] + test_eval['L_SCORE']

for season in sorted(test_eval.SEASON.unique()):
    sub = test_eval[test_eval.SEASON == season]
    print(f"{season}  spread MAE: {mean_absolute_error(sub.ACTUAL_SPREAD, sub.PRED_SPREAD):.2f} pts  "
          f"total MAE: {mean_absolute_error(sub.ACTUAL_TOTAL, sub.PRED_TOTAL):.2f} pts")

print(f"Overall spread MAE: {mean_absolute_error(test_eval.ACTUAL_SPREAD, test_eval.PRED_SPREAD):.2f} pts")
print(f"Overall total  MAE: {mean_absolute_error(test_eval.ACTUAL_TOTAL,  test_eval.PRED_TOTAL):.2f} pts")

Spread and total (no-seed) models saved.
2024  spread MAE: 11.76 pts  total MAE: 11.99 pts
2025  spread MAE: 15.28 pts  total MAE: 13.20 pts
Overall spread MAE: 13.55 pts
Overall total  MAE: 12.61 pts


C:\Users\joebu\AppData\Local\Temp\ipykernel_31904\3173959307.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_eval['PRED_SPREAD'] = optimal_spread_model.predict(test_eval[feature_cols])
C:\Users\joebu\AppData\Local\Temp\ipykernel_31904\3173959307.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_eval['PRED_TOTAL']  = optimal_total_model.predict(test_eval[feature_cols])
C:\Users\joebu\AppData\Local\Temp\ipykernel_31904\3173959307.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the resul

In [14]:
import os
os.makedirs('model', exist_ok=True)

optimal_model = all_rounds.best_estimator_
joblib.dump(optimal_model, 'model/march_madness_model_no_seed_W.joblib')
print(f'Win model (no-seed) saved. Best params: {all_rounds.best_params_}')
print(f'Test seasons: {sorted(test_eval.SEASON.unique())}')

Win model (no-seed) saved. Best params: {'learning_rate': 0.2, 'max_depth': 3, 'min_child_weight': 2, 'n_estimators': 100}
Test seasons: [np.int64(2024), np.int64(2025)]


import os
os.makedirs('model', exist_ok=True)

optimal_model = all_rounds.best_estimator_
joblib.dump(optimal_model, 'model/march_madness_model_W.joblib')
print(f'Model saved. Best params: {all_rounds.best_params_}')
print(f'Test seasons: {sorted(test_eval.SEASON.unique())}')

In [15]:
def predict_games(win_model, spread_model, total_model, df, feature_cols):
    """Run all three model predictions, filling missing feature cols with 0."""
    df = df.copy()
    for col in feature_cols:
        if col not in df.columns:
            df[col] = 0
    df['PRED_WIN_INDICATOR'] = win_model.predict(df[feature_cols])
    df['PRED_SPREAD'] = spread_model.predict(df[feature_cols]).round(1)
    df['PRED_TOTAL']  = total_model.predict(df[feature_cols]).round(1)
    return df


def add_team_names(result, teams):
    """Join team names onto a result dataframe that has W_TEAMID and L_TEAMID."""
    result = result.merge(
        teams[['TEAMID', 'TEAMNAME']], left_on='L_TEAMID', right_on='TEAMID'
    ).rename(columns={'TEAMNAME': 'L_TEAM_NAME'}).drop(columns=['TEAMID'])
    result = result.merge(
        teams[['TEAMID', 'TEAMNAME']], left_on='W_TEAMID', right_on='TEAMID'
    ).rename(columns={'TEAMNAME': 'W_TEAM_NAME'}).drop(columns=['TEAMID'])
    return result


def get_round_winners(result):
    """Pick the predicted winner from each game, returning (W_TEAM_ID, W_TEAM_NAME, W_SEED, W_REGION)."""
    r = result.copy()
    r['W_TEAM_ID']   = np.where(r.PRED_WIN_INDICATOR == 1, r.W_TEAMID,    r.L_TEAMID)
    r['W_TEAM_NAME'] = np.where(r.PRED_WIN_INDICATOR == 1, r.W_TEAM_NAME, r.L_TEAM_NAME)
    r['W_SEED']      = np.where(r.PRED_WIN_INDICATOR == 1, r.W_SEED,      r.L_SEED)
    r['W_REGION']    = np.where(r.PRED_WIN_INDICATOR == 1, r.W_REGION,    r.L_REGION)
    return r[['W_TEAM_ID', 'W_TEAM_NAME', 'W_SEED', 'W_REGION']]


def add_season_stats(matchups, season_stats, season_year):
    """Join season stats onto matchups that have WTEAMID2 and LTEAMID2 columns."""
    season = season_stats[season_stats.SEASON == season_year].drop(columns=['SEASON'])
    season = season.drop(columns=[c for c in ['REGION'] if c in season.columns])
    season_w = season.copy()
    season_w.columns = ['W_' + c for c in season_w.columns]
    season_l = season.copy()
    season_l.columns = ['L_' + c for c in season_l.columns]

    df = matchups.merge(season_w, left_on='WTEAMID2', right_on='W_TEAMID').drop(columns=['W_TEAMID'])
    df = df.merge(season_l, left_on='LTEAMID2', right_on='L_TEAMID').drop(columns=['L_TEAMID'])
    df = df.rename(columns={'WTEAMID2': 'W_TEAMID', 'LTEAMID2': 'L_TEAMID'})
    return df


teams = pd.read_csv('data_2026/WTeams.csv')
teams.columns = teams.columns.str.upper()

season_stats = pd.read_csv('data_2026/final_season_stats_W.csv')

In [16]:
# Load seeds for the bracket season
seeds_raw = pd.read_csv('data_2026/WNCAATourneySeeds.csv')
seeds_raw.columns = seeds_raw.columns.str.upper()
seeds_bracket = seeds_raw[seeds_raw.SEASON == BRACKET_SEASON].copy()
seeds_bracket['REGION'] = seeds_bracket['SEED'].str[0]
seeds_bracket['IS_PLAYIN'] = seeds_bracket['SEED'].str[1:].str.contains('[ab]', regex=True)
seeds_bracket['SEED_NUM'] = seeds_bracket['SEED'].str[1:].str.replace('[a-z]', '', regex=True).astype(int)

playin_seeds = seeds_bracket[seeds_bracket.IS_PLAYIN][['TEAMID', 'REGION', 'SEED_NUM']].rename(columns={'SEED_NUM': 'SEED'})
regular_seeds = seeds_bracket[~seeds_bracket.IS_PLAYIN][['TEAMID', 'REGION', 'SEED_NUM']].rename(columns={'SEED_NUM': 'SEED'})

playin_rows = []
for (region, seed), group in playin_seeds.groupby(['REGION', 'SEED']):
    team_ids = group.TEAMID.tolist()
    if len(team_ids) == 2:
        playin_rows.append({
            'WTEAMID2': team_ids[0], 'LTEAMID2': team_ids[1],
            'W_SEED': seed, 'L_SEED': seed,
            'W_REGION': region, 'L_REGION': region,
        })

if playin_rows:
    playin_matchups = pd.DataFrame(playin_rows)
    games_playin = add_season_stats(playin_matchups, season_stats, BRACKET_SEASON)
    result_playin = predict_games(optimal_model, optimal_spread_model, optimal_total_model, games_playin, feature_cols)
    result_playin = add_team_names(result_playin, teams)
    playin_result = result_playin[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED', 'L_REGION', 'W_TEAM_NAME', 'W_REGION', 'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']]
    playin_winners = get_round_winners(playin_result)
    print(f'Play-in results ({BRACKET_SEASON}):')
    display(playin_result[['W_TEAM_NAME', 'L_TEAM_NAME', 'W_SEED', 'W_REGION', 'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']].sort_values('W_REGION'))
else:
    playin_winners = pd.DataFrame(columns=['W_TEAM_ID', 'W_TEAM_NAME', 'W_SEED', 'W_REGION'])
    print(f'No play-in games found for {BRACKET_SEASON}')

Play-in results (2026):


,W_TEAM_NAME,L_TEAM_NAME,W_SEED,W_REGION,PRED_WIN_INDICATOR,PRED_SPREAD,PRED_TOTAL
0,Arizona St,Virginia,10,X,0,-8.3,129.000000
1,Samford,Southern Univ,16,X,0,0.5,115.099998
2,Missouri St,SF Austin,16,Y,0,-6.5,132.199997
3,Nebraska,Richmond,11,Z,1,12.3,139.699997


## Round of 64

In [17]:
ROUND_1_PAIRS = [(1, 16), (8, 9), (5, 12), (4, 13), (6, 11), (3, 14), (7, 10), (2, 15)]

round1_seeds = regular_seeds.copy()
if not playin_winners.empty:
    playin_w_seeds = playin_winners.rename(
        columns={'W_TEAM_ID': 'TEAMID', 'W_SEED': 'SEED', 'W_REGION': 'REGION'}
    )[['TEAMID', 'REGION', 'SEED']]
    round1_seeds = pd.concat([round1_seeds, playin_w_seeds], ignore_index=True)

round1_rows = []
for region in round1_seeds.REGION.unique():
    r = round1_seeds[round1_seeds.REGION == region].set_index('SEED')
    for s1, s2 in ROUND_1_PAIRS:
        if s1 in r.index and s2 in r.index:
            round1_rows.append({
                'WTEAMID2': r.loc[s1, 'TEAMID'], 'LTEAMID2': r.loc[s2, 'TEAMID'],
                'W_SEED': s1, 'L_SEED': s2,
                'W_REGION': region, 'L_REGION': region,
            })

round1_matchups = pd.DataFrame(round1_rows)
games_r1 = add_season_stats(round1_matchups, season_stats, BRACKET_SEASON)
result = predict_games(optimal_model, optimal_spread_model, optimal_total_model, games_r1, feature_cols)
result = add_team_names(result, teams)
print(f'Round of 64 — {BRACKET_SEASON}: {len(result)} games')

Round of 64 — 2026: 32 games


In [18]:
round_1_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED', 'L_REGION', 'W_TEAM_NAME', 'W_REGION', 'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']]
round_1_results.sort_values('W_REGION')

,L_TEAMID,W_TEAMID,L_TEAM_NAME,L_SEED,W_SEED,L_REGION,W_TEAM_NAME,W_REGION,PRED_WIN_INDICATOR,PRED_SPREAD,PRED_TOTAL
0,3427,3163,UT San Antonio,16,1,W,Connecticut,W,1,53.500000,122.800003
1,3393,3235,Syracuse,9,8,W,Iowa St,W,0,-0.800000,142.000000
2,3293,3268,Murray St,12,5,W,Maryland,W,1,21.400000,150.300003
3,3442,3314,W Illinois,13,4,W,North Carolina,W,1,20.700001,142.000000
4,3193,3323,Fairfield,11,6,W,Notre Dame,W,0,12.300000,132.399994
5,3224,3326,Howard,14,3,W,Ohio St,W,1,22.600000,139.500000
6,3160,3228,Colorado,10,7,W,Illinois,W,1,4.100000,131.000000
7,3219,3435,High Point,15,2,W,Vanderbilt,W,1,18.700001,154.000000
14,3438,3208,Virginia,10,7,X,Georgia,X,0,-1.400000,133.100006
13,3471,3395,UC San Diego,14,3,X,TCU,X,1,26.500000,140.800003


# Winners round 1

In [19]:
round_1_winners = get_round_winners(round_1_results)
round_1_winners.sort_values('W_REGION')

,W_TEAM_ID,W_TEAM_NAME,W_SEED,W_REGION
0,3163,Connecticut,1,W
1,3393,Syracuse,9,W
2,3268,Maryland,5,W
3,3314,North Carolina,4,W
4,3193,Fairfield,11,W
5,3326,Ohio St,3,W
6,3228,Illinois,7,W
7,3435,Vanderbilt,2,W
14,3438,Virginia,10,X
13,3395,TCU,3,X


In [20]:
ROUND_2_SEED_PAIRS = [
    (1,8),(1,9),(16,8),(16,9),
    (4,5),(4,12),(13,5),(13,12),
    (3,6),(3,11),(14,6),(14,11),
    (2,7),(2,10),(15,7),(15,10),
]

df1 = round_1_winners.rename(columns={'W_TEAM_NAME':'W_TEAM_NAME','W_REGION':'W_REGION','W_TEAM_ID':'WTEAMID2','W_SEED':'W_SEED'})
df2 = round_1_winners.rename(columns={'W_TEAM_NAME':'L_TEAM_NAME','W_REGION':'L_REGION','W_TEAM_ID':'LTEAMID2','W_SEED':'L_SEED'})

cross = df1.merge(df2, how='cross', suffixes=('','_r'))
seed_set = set(ROUND_2_SEED_PAIRS)
mask = (
    (cross.W_REGION == cross.L_REGION) &
    cross.apply(lambda r: (r.W_SEED, r.L_SEED) in seed_set, axis=1)
)
second_round_matchups = cross[mask][['W_TEAM_NAME','W_REGION','WTEAMID2','W_SEED','L_TEAM_NAME','L_REGION','LTEAMID2','L_SEED']]
second_round_matchups.sort_values('W_REGION')

,W_TEAM_NAME,W_REGION,WTEAMID2,W_SEED,L_TEAM_NAME,L_REGION,LTEAMID2,L_SEED
1,Connecticut,W,3163,1,Syracuse,W,3393,9
98,North Carolina,W,3314,4,Maryland,W,3268,5
164,Ohio St,W,3326,3,Fairfield,W,3193,11
230,Vanderbilt,W,3435,2,Illinois,W,3228,7
265,South Carolina,X,3376,1,USC,X,3425,9
362,Oklahoma,X,3328,4,Michigan St,X,3277,5
428,TCU,X,3395,3,Washington,X,3449,6
494,Iowa,X,3234,2,Virginia,X,3438,10
529,Texas,Y,3400,1,Virginia Tech,Y,3439,9
626,West Virginia,Y,3452,4,Kentucky,Y,3246,5


# ROUND OF 32!!!

In [21]:
second_round_matchups[['W_TEAM_NAME','L_TEAM_NAME','W_SEED','L_SEED','W_REGION']].rename(
    columns={'W_TEAM_NAME':'TEAM_1','L_TEAM_NAME':'TEAM_2','W_SEED':'SEED_1','L_SEED':'SEED_2','W_REGION':'REGION'}
).sort_values('REGION')

,TEAM_1,TEAM_2,SEED_1,SEED_2,REGION
1,Connecticut,Syracuse,1,9,W
98,North Carolina,Maryland,4,5,W
164,Ohio St,Fairfield,3,11,W
230,Vanderbilt,Illinois,2,7,W
265,South Carolina,USC,1,9,X
362,Oklahoma,Michigan St,4,5,X
428,TCU,Washington,3,6,X
494,Iowa,Virginia,2,10,X
529,Texas,Virginia Tech,1,9,Y
626,West Virginia,Kentucky,4,5,Y


In [22]:
games_32 = add_season_stats(second_round_matchups[['WTEAMID2', 'LTEAMID2', 'W_SEED', 'L_SEED', 'W_REGION', 'L_REGION']], season_stats, BRACKET_SEASON)
result = predict_games(optimal_model, optimal_spread_model, optimal_total_model, games_32, feature_cols)
result = add_team_names(result, teams)

round_2_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED', 'L_REGION', 'W_TEAM_NAME', 'W_REGION', 'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']]
round_2_results

,L_TEAMID,W_TEAMID,L_TEAM_NAME,L_SEED,W_SEED,L_REGION,W_TEAM_NAME,W_REGION,PRED_WIN_INDICATOR,PRED_SPREAD,PRED_TOTAL
0,3393,3163,Syracuse,9,1,W,Connecticut,W,1,19.500000,145.600006
1,3268,3314,Maryland,5,4,W,North Carolina,W,0,-3.800000,140.300003
2,3193,3326,Fairfield,11,3,W,Ohio St,W,0,9.300000,142.600006
3,3228,3435,Illinois,7,2,W,Vanderbilt,W,1,5.100000,146.600006
4,3425,3376,USC,9,1,X,South Carolina,X,1,29.500000,145.000000
5,3277,3328,Michigan St,5,4,X,Oklahoma,X,1,5.000000,152.000000
6,3449,3395,Washington,6,3,X,TCU,X,1,7.100000,137.300003
7,3438,3234,Virginia,10,2,X,Iowa,X,0,2.800000,135.899994
8,3439,3400,Virginia Tech,9,1,Y,Texas,Y,1,12.500000,147.800003
9,3246,3452,Kentucky,5,4,Y,West Virginia,Y,0,5.700000,128.800003


In [23]:
round_2_winners = get_round_winners(round_2_results)
round_2_winners.sort_values('W_REGION')

,W_TEAM_ID,W_TEAM_NAME,W_SEED,W_REGION
0,3163,Connecticut,1,W
1,3268,Maryland,5,W
2,3193,Fairfield,11,W
3,3435,Vanderbilt,2,W
4,3376,South Carolina,1,X
5,3328,Oklahoma,4,X
6,3395,TCU,3,X
7,3438,Virginia,10,X
8,3400,Texas,1,Y
9,3246,Kentucky,5,Y


# Round 2 winners

In [24]:
ROUND_3_SEED_PAIRS = [
    (s1, s2)
    for s1 in [1,16,8,9]
    for s2 in [5,12,4,13]
] + [
    (s1, s2)
    for s1 in [6,11,3,14]
    for s2 in [7,10,2,15]
]

df1 = round_2_winners.rename(columns={'W_TEAM_NAME':'W_TEAM_NAME','W_REGION':'W_REGION','W_TEAM_ID':'WTEAMID2','W_SEED':'W_SEED'})
df2 = round_2_winners.rename(columns={'W_TEAM_NAME':'L_TEAM_NAME','W_REGION':'L_REGION','W_TEAM_ID':'LTEAMID2','W_SEED':'L_SEED'})

cross = df1.merge(df2, how='cross', suffixes=('','_r'))
seed_set = set(ROUND_3_SEED_PAIRS)
mask = (
    (cross.W_REGION == cross.L_REGION) &
    cross.apply(lambda r: (r.W_SEED, r.L_SEED) in seed_set, axis=1)
)
third_round_matchups = cross[mask][['W_TEAM_NAME','W_REGION','WTEAMID2','W_SEED','L_TEAM_NAME','L_REGION','LTEAMID2','L_SEED']]
third_round_matchups.sort_values('W_REGION')

,W_TEAM_NAME,W_REGION,WTEAMID2,W_SEED,L_TEAM_NAME,L_REGION,LTEAMID2,L_SEED
1,Connecticut,W,3163,1,Maryland,W,3268,5
35,Fairfield,W,3193,11,Vanderbilt,W,3435,2
69,South Carolina,X,3376,1,Oklahoma,X,3328,4
103,TCU,X,3395,3,Virginia,X,3438,10
137,Texas,Y,3400,1,Kentucky,Y,3246,5
171,Louisville,Y,3257,3,Michigan,Y,3276,2
205,UCLA,Z,3417,1,Mississippi,Z,3279,5
239,Duke,Z,3181,3,LSU,Z,3261,2


# SWEET SIXTEEN!!!

In [25]:
third_round_matchups[['W_TEAM_NAME','L_TEAM_NAME','W_SEED','L_SEED','W_REGION']].rename(
    columns={'W_TEAM_NAME':'TEAM_1','L_TEAM_NAME':'TEAM_2','W_SEED':'SEED_1','L_SEED':'SEED_2','W_REGION':'REGION'}
).sort_values('REGION')

,TEAM_1,TEAM_2,SEED_1,SEED_2,REGION
1,Connecticut,Maryland,1,5,W
35,Fairfield,Vanderbilt,11,2,W
69,South Carolina,Oklahoma,1,4,X
103,TCU,Virginia,3,10,X
137,Texas,Kentucky,1,5,Y
171,Louisville,Michigan,3,2,Y
205,UCLA,Mississippi,1,5,Z
239,Duke,LSU,3,2,Z


In [26]:
games_16 = add_season_stats(third_round_matchups[['WTEAMID2', 'LTEAMID2', 'W_SEED', 'L_SEED', 'W_REGION', 'L_REGION']], season_stats, BRACKET_SEASON)
result = predict_games(optimal_model, optimal_spread_model, optimal_total_model, games_16, feature_cols)
result = add_team_names(result, teams)

sweet_16_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED', 'L_REGION', 'W_TEAM_NAME', 'W_REGION', 'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']]
sweet_16_winners = get_round_winners(sweet_16_results)
sweet_16_results

,L_TEAMID,W_TEAMID,L_TEAM_NAME,L_SEED,W_SEED,L_REGION,W_TEAM_NAME,W_REGION,PRED_WIN_INDICATOR,PRED_SPREAD,PRED_TOTAL
0,3268,3163,Maryland,5,1,W,Connecticut,W,1,10.4,138.500000
1,3435,3193,Vanderbilt,2,11,W,Fairfield,W,0,-4.1,158.000000
2,3328,3376,Oklahoma,4,1,X,South Carolina,X,1,7.4,156.800003
3,3438,3395,Virginia,10,3,X,TCU,X,1,6.9,139.500000
4,3246,3400,Kentucky,5,1,Y,Texas,Y,1,20.0,144.399994
5,3276,3257,Michigan,2,3,Y,Louisville,Y,1,-1.2,149.300003
6,3279,3417,Mississippi,5,1,Z,UCLA,Z,0,6.4,141.100006
7,3261,3181,LSU,2,3,Z,Duke,Z,0,-10.4,144.300003


# Sweet 16 winners!

In [27]:
ELITE_8_SEED_PAIRS = [
    (s1, s2)
    for s1 in [1,16,8,9,5,12,4,13]
    for s2 in [6,11,3,14,7,10,2,15]
]

df1 = sweet_16_winners.rename(columns={'W_TEAM_NAME':'W_TEAM_NAME','W_REGION':'W_REGION','W_TEAM_ID':'WTEAMID2','W_SEED':'W_SEED'})
df2 = sweet_16_winners.rename(columns={'W_TEAM_NAME':'L_TEAM_NAME','W_REGION':'L_REGION','W_TEAM_ID':'LTEAMID2','W_SEED':'L_SEED'})

cross = df1.merge(df2, how='cross', suffixes=('','_r'))
seed_set = set(ELITE_8_SEED_PAIRS)
mask = (
    (cross.W_REGION == cross.L_REGION) &
    cross.apply(lambda r: (r.W_SEED, r.L_SEED) in seed_set, axis=1)
)
elite_8_matchups = cross[mask][['W_TEAM_NAME','W_REGION','WTEAMID2','W_SEED','L_TEAM_NAME','L_REGION','LTEAMID2','L_SEED']]
elite_8_matchups.sort_values('W_REGION')

,W_TEAM_NAME,W_REGION,WTEAMID2,W_SEED,L_TEAM_NAME,L_REGION,LTEAMID2,L_SEED
1,Connecticut,W,3163,1,Vanderbilt,W,3435,2
19,South Carolina,X,3376,1,TCU,X,3395,3
37,Texas,Y,3400,1,Louisville,Y,3257,3
55,Mississippi,Z,3279,5,LSU,Z,3261,2


# Elite 8

In [28]:
elite_8_matchups[['W_TEAM_NAME','L_TEAM_NAME','W_SEED','L_SEED','W_REGION']].rename(
    columns={'W_TEAM_NAME':'TEAM_1','L_TEAM_NAME':'TEAM_2','W_SEED':'SEED_1','L_SEED':'SEED_2','W_REGION':'REGION'}
).sort_values('REGION')

,TEAM_1,TEAM_2,SEED_1,SEED_2,REGION
1,Connecticut,Vanderbilt,1,2,W
19,South Carolina,TCU,1,3,X
37,Texas,Louisville,1,3,Y
55,Mississippi,LSU,5,2,Z


In [29]:
elite_8 = add_season_stats(elite_8_matchups[['WTEAMID2', 'LTEAMID2', 'W_SEED', 'L_SEED', 'W_REGION', 'L_REGION']], season_stats, BRACKET_SEASON)
result = predict_games(optimal_model, optimal_spread_model, optimal_total_model, elite_8, feature_cols)
result = add_team_names(result, teams)

elite_8_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED', 'L_REGION', 'W_TEAM_NAME', 'W_REGION', 'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']]
elite_8_winners = get_round_winners(elite_8_results)
elite_8_results

,L_TEAMID,W_TEAMID,L_TEAM_NAME,L_SEED,W_SEED,L_REGION,W_TEAM_NAME,W_REGION,PRED_WIN_INDICATOR,PRED_SPREAD,PRED_TOTAL
0,3435,3163,Vanderbilt,2,1,W,Connecticut,W,1,15.6,148.699997
1,3395,3376,TCU,3,1,X,South Carolina,X,1,14.5,145.100006
2,3257,3400,Louisville,3,1,Y,Texas,Y,1,10.0,147.800003
3,3261,3279,LSU,2,5,Z,Mississippi,Z,0,-7.2,145.600006


# Elite 8 winners

In [30]:
# Final four matchups: W vs X regions, Y vs Z regions
wx = elite_8_winners[elite_8_winners.W_REGION == 'W'].merge(
    elite_8_winners[elite_8_winners.W_REGION == 'X'].rename(
        columns={'W_TEAM_ID':'TEAM_2','W_TEAM_NAME':'W_TEAM_NAME_2','W_SEED':'SEED_2','W_REGION':'REGION_2'}
    ),
    how='cross',
)

yz = elite_8_winners[elite_8_winners.W_REGION == 'Y'].merge(
    elite_8_winners[elite_8_winners.W_REGION == 'Z'].rename(
        columns={'W_TEAM_ID':'TEAM_2','W_TEAM_NAME':'W_TEAM_NAME_2','W_SEED':'SEED_2','W_REGION':'REGION_2'}
    ),
    how='cross',
)

final_four_matchups = pd.concat([wx, yz]).rename(columns={
    'W_TEAM_ID': 'WTEAMID2',
    'W_TEAM_NAME_2': 'L_TEAM_NAME',
    'REGION_2': 'L_REGION',
    'TEAM_2': 'LTEAMID2',
    'SEED_2': 'L_SEED',
})[['W_TEAM_NAME','W_REGION','WTEAMID2','W_SEED','L_TEAM_NAME','L_REGION','LTEAMID2','L_SEED']]

# FINAL FOUR!

In [31]:
final_four_matchups[['W_TEAM_NAME','L_TEAM_NAME','W_SEED','L_SEED','W_REGION']].rename(
    columns={'W_TEAM_NAME':'TEAM_1','L_TEAM_NAME':'TEAM_2','W_SEED':'SEED_1','L_SEED':'SEED_2','W_REGION':'REGION'}
).sort_values('REGION')

,TEAM_1,TEAM_2,SEED_1,SEED_2,REGION
0,Connecticut,South Carolina,1,1,W
0,Texas,LSU,1,2,Y


In [32]:
final_four = add_season_stats(final_four_matchups[['WTEAMID2', 'LTEAMID2', 'W_SEED', 'L_SEED', 'W_REGION', 'L_REGION']], season_stats, BRACKET_SEASON)
result = predict_games(optimal_model, optimal_spread_model, optimal_total_model, final_four, feature_cols)
result = add_team_names(result, teams)

final_four_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED', 'L_REGION', 'W_TEAM_NAME', 'W_REGION', 'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']]
final_four_winners = get_round_winners(final_four_results)
final_four_results

,L_TEAMID,W_TEAMID,L_TEAM_NAME,L_SEED,W_SEED,L_REGION,W_TEAM_NAME,W_REGION,PRED_WIN_INDICATOR,PRED_SPREAD,PRED_TOTAL
0,3376,3163,South Carolina,1,1,X,Connecticut,W,0,-0.3,145.800003
1,3261,3400,LSU,2,1,Z,Texas,Y,1,5.2,146.600006


In [33]:
# Championship: cross-join final four winners and keep the one unique matchup
championship_matchup = (
    final_four_winners.rename(columns={'W_TEAM_ID':'WTEAMID2','W_TEAM_NAME':'W_TEAM_NAME','W_SEED':'W_SEED','W_REGION':'W_REGION'})
    .merge(
        final_four_winners.rename(columns={'W_TEAM_ID':'LTEAMID2','W_TEAM_NAME':'L_TEAM_NAME','W_SEED':'L_SEED','W_REGION':'L_REGION'}),
        how='cross',
    )
    .query('WTEAMID2 != LTEAMID2')
    .head(1)
)

# CHAMPIONSHIP!

In [34]:
championship_matchup[['W_TEAM_NAME','L_TEAM_NAME','W_SEED','L_SEED','W_REGION']].rename(
    columns={'W_TEAM_NAME':'TEAM_1','L_TEAM_NAME':'TEAM_2','W_SEED':'SEED_1','L_SEED':'SEED_2','W_REGION':'REGION'}
)

,TEAM_1,TEAM_2,SEED_1,SEED_2,REGION
1,South Carolina,Texas,1,1,X


In [35]:
championship = add_season_stats(championship_matchup[['WTEAMID2', 'LTEAMID2', 'W_SEED', 'L_SEED', 'W_REGION', 'L_REGION']], season_stats, BRACKET_SEASON)
result = predict_games(optimal_model, optimal_spread_model, optimal_total_model, championship, feature_cols)
result = add_team_names(result, teams)

championship_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED', 'L_REGION', 'W_TEAM_NAME', 'W_REGION', 'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']]
championship_results

,L_TEAMID,W_TEAMID,L_TEAM_NAME,L_SEED,W_SEED,L_REGION,W_TEAM_NAME,W_REGION,PRED_WIN_INDICATOR,PRED_SPREAD,PRED_TOTAL
0,3400,3376,Texas,1,1,Y,South Carolina,X,0,-1.6,147.5


In [36]:
champion = get_round_winners(championship_results)
champion

,W_TEAM_ID,W_TEAM_NAME,W_SEED,W_REGION
0,3400,Texas,1,Y


# CHAMPION!

## Getting started with 2026 predictions

Before running the bracket prediction cells, you need 2026 tournament seed data in `data_2026/MNCAATourneySeeds.csv`.

Once the seeds are added, re-run `1_cleaning_2026.ipynb` to regenerate `final_season_stats.csv` with 2026 season data, then run this notebook top to bottom.